# A8: A correction, a stray file, and a repeatable result
**IT 630 · Friday, October 23, 2026 · virtual · about 50 minutes**

Dana needs to correct a Pop-Tart report without losing the original answer.
She also finds a Parquet file someone copied into the table folder. Should
that file change the report? And if the correction job runs twice, should
sales double?

In A7 you built a star and a tiny lakehouse. Today you use the same audited
export to test the difference between files on disk and a committed table,
then trace a correction across table versions. This connects the lakehouse to
this week's promise: a completed pipeline still needs evidence that its result
is the intended one.

**First:** File → Save a copy in Drive. Work in that copy.
**▶️ Run:** supplied code. **🧩 Fill-in:** replace the marked `None` before running.
**✏️ Answer:** Q1, Q3, and Q5 are one-sentence predictions: write each before
the next run. Q2, Q4, and Q6 explain the output afterward; keep them brief.
A wrong prediction is fine. Do not use Run all. Keep cell outputs saved;
you do not need to copy their numbers into Q2 or Q4.

Everything is a local miniature in your Colab session, not shared durable
storage. No A7 session or saved files are needed. Setup rebuilds it for you.
Submit an Anyone with the link / Viewer Colab URL in Canvas, with the version
fill-in run and every answer completed. Do not upload a file.

## 0. Rebuild and read the supplied recap (7 minutes)
**Supplied recap (no answer needed):** one fact row is one date/store/SKU
measurement; `(date, store_id, sku)` is the unique key. The per-store report
uses `SELECT store_id ... GROUP BY store_id`, summing across the storm dates
and selected SKU. Both decisions are supplied from A7. We keep product descriptions
in a dimension and measurements in a fact table. To make the file experiment
visible, this small fact table uses one unpartitioned file rather than A7's
many tiny date partitions. This is a teaching choice, not a storage recommendation.
The first cell installs the same Delta library as A7. Wait for it to finish.

In [ ]:
%pip install -q deltalake==1.6.6

import json
import shutil
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import duckdb
from deltalake import DeltaTable, write_deltalake
print("ready")

In [ ]:
# ▶️ Run: this is the RESET cell. Re-running it erases today's local experiments.
raw = pd.read_csv("https://blkfin.github.io/it630/fall2026/activities/data/poptart-sales-2026-09-10-full.csv")
fact_columns = ["date", "store_id", "sku", "units", "unit_price"]
base_fact = raw[fact_columns].copy()
product = raw[["sku", "product_name", "flavor"]].drop_duplicates().reset_index(drop=True)
ROOT = Path("a8_lakehouse")
FACT = ROOT / "fact_sales"
DIM = ROOT / "dim_product"
shutil.rmtree(ROOT, ignore_errors=True)
write_deltalake(str(FACT), base_fact)
write_deltalake(str(DIM), product)

# Each call reads a specific committed snapshot; no cached table object.
def snapshot(version=None):
    return DeltaTable(str(FACT), version=version).to_pandas()

def storm_total(frame):
    selected = frame[(frame.sku == "PT-STRW") & frame.date.between("2026-09-05", "2026-09-09")]
    return int(selected.units.sum())

print("fact rows:", len(snapshot()), "dimension rows:", len(product))
print("fact version:", DeltaTable(str(FACT)).version())
print("baseline storm units:", storm_total(snapshot()))

## 1. A file arrives without a commit (8 minutes)
Someone copies a committed Parquet file into the same folder under a new name.
The bytes are identical; no Delta write is performed. The next cell overwrites
that same stray filename on a retry, so it does not accumulate copies.

**✏️ Q1. Predict in one sentence: will a plain Parquet glob and a Delta reader see the same row count and storm total after the copy, and why?**

_Your answer: double-click and type here._

In [ ]:
# ▶️ Run: copy one file WITHOUT adding it to the transaction log.
original_file = Path(DeltaTable(str(FACT), version=0).file_uris()[0])
stray_file = FACT / "stray-copy.parquet"
shutil.copyfile(original_file, stray_file)

plain_files = pd.concat([pq.read_table(path).to_pandas() for path in sorted(FACT.glob("*.parquet"))], ignore_index=True)
committed = snapshot()
print("plain glob:", len(plain_files), "rows; storm units:", storm_total(plain_files))
print("Delta:", len(committed), "rows; storm units:", storm_total(committed))
print("Delta version:", DeltaTable(str(FACT)).version())
print("Delta active files:", [Path(uri).name for uri in DeltaTable(str(FACT)).file_uris()])

**✏️ Q2. Is the stray file part of the Delta table? Name the printed line that shows it, explain any difference from your prediction, and say which reader Dana should use.**

_Your answer: double-click and type here._

## 2. Commit a deliberate correction (10 minutes)
For this authored exercise, Dana has verified that the September 5, S05,
PT-STRW record needs **10 additional units**. This is a scenario correction,
not a claim that the course's audited source file was wrong.

We retain the raw source unchanged, prepare a corrected fact snapshot from
that fixed source, check its grain and units, and replace the committed table.
This is a **full-snapshot rebuild**: overwrite the table with the complete
intended snapshot from fixed raw input plus the correction rule. An
**incremental upsert** instead applies incoming new or changed records by key
to an existing table; it would need its own retry/deduplication rules. We do
not implement that operation here.
The product dimension does not change.

Before running: write your prediction below. The correction cell is safe to
repeat: it always starts from `base_fact`, never from already corrected data.
Each repeat adds a version; report the versions you actually see.

**✏️ Q3. Predict in one sentence: how will row count, storm total, S05 storm total, and the fact and dimension versions change, and why?**

_Your answer: double-click and type here._

In [ ]:
# ▶️ Run: construct the intended snapshot from FIXED raw input and rules.
def corrected_snapshot():
    corrected = base_fact.copy(deep=True)
    target = ((corrected.date == "2026-09-05") &
              (corrected.store_id == "S05") & (corrected.sku == "PT-STRW"))
    assert int(target.sum()) == 1, "The correction must identify exactly one fact row"
    corrected.loc[target, "units"] = corrected.loc[target, "units"] + 10
    assert not corrected.duplicated(["date", "store_id", "sku"]).any()
    assert corrected.units.notna().all() and corrected.units.ge(0).all()
    return corrected

write_deltalake(str(FACT), corrected_snapshot(), mode="overwrite")
corrected = snapshot()
print("fact version:", DeltaTable(str(FACT)).version())
print("dimension version:", DeltaTable(str(DIM)).version())
print("rows:", len(corrected), "storm units:", storm_total(corrected))

In [ ]:
# ▶️ Run: supplied recap query, one row per store across the storm dates.
by_store = duckdb.sql("""
    SELECT store_id AS store, SUM(units) AS strawberry_units
    FROM corrected
    WHERE sku = 'PT-STRW' AND date BETWEEN '2026-09-05' AND '2026-09-09'
    GROUP BY store_id
    ORDER BY strawberry_units DESC, store
""").df()
print(by_store.to_string(index=False))

**✏️ Q4. Compare the output with your prediction: why did the totals change while the row count stayed the same, and why did only one table gain a version?**

_Your answer: double-click and type here._

## 3. Read the original and repeat the job (11 minutes)
**Time travel** means reading a named earlier table version. It does not undo
a correction or change the latest version. Old files and log entries must
still be available; retention cleanup can remove that ability.

Fill in the original fact version number below. Then compare the old answer
with the current answer before you retry the correction.

In [ ]:
# 🧩 Fill-in: replace None with the original version number (an integer).
ORIGINAL_VERSION = None
if not isinstance(ORIGINAL_VERSION, int) or isinstance(ORIGINAL_VERSION, bool) or ORIGINAL_VERSION < 0:
    raise ValueError("Set ORIGINAL_VERSION to a nonnegative whole number (no quotes), then rerun this cell")
original = snapshot(version=ORIGINAL_VERSION)
print("original storm units:", storm_total(original))
print("current storm units:", storm_total(snapshot()))
print("latest fact version after historical read:", DeltaTable(str(FACT)).version())

**✏️ Q5. Predict in one sentence: what row count and storm total will the SAME correction produce again, and would adding 10 to the latest snapshot instead give the same result?**

_Your answer: double-click and type here._

In [ ]:
# ▶️ Run: the same fixed input and correction rule, a second committed write.
write_deltalake(str(FACT), corrected_snapshot(), mode="overwrite")
retried = snapshot()
print("fact version:", DeltaTable(str(FACT)).version())
print("rows:", len(retried), "storm units:", storm_total(retried))
print("same table content:", corrected.sort_values(["date", "store_id", "sku"]).reset_index(drop=True).equals(
    retried.sort_values(["date", "store_id", "sku"]).reset_index(drop=True)))

# Read the changes recorded by the first correction, not all physical files.
actions = [json.loads(line) for line in (FACT / "_delta_log" / "00000000000000000001.json").read_text().splitlines()]
print("correction action types:", sorted({next(iter(action)) for action in actions}))
print("original still readable:", storm_total(snapshot(version=0)))

**✏️ Q6. Compare prediction and observation. How can the version change while the content stays the same? Define idempotence in terms of the intended table content, and explain why this rebuild has it.**

_Your answer: double-click and type here._

## 4. Evidence for Dana and submit (9 minutes)
A Delta commit records a table change; our checks test selected properties of
the data. Neither substitutes for the other. A wrong but nonnegative correction
could pass our checks and still be committed. A plain file glob is especially
unsafe after overwrites: old, removed, and uncommitted files can remain on disk.
Do not delete old files by hand to make a glob agree with the log.

**✏️ Q7. Using today’s saved outputs, give Dana a short evidence note: cite the current fact/dimension versions, original/current totals, and retry content comparison. Explain what the successful correction commit establishes about active table membership (contrast the stray file), what one named check establishes, and one factual claim neither the commit nor that check proves. Did reading version 0 undo the correction? Cite the latest-version output.**

_Your answer: double-click and type here._

**Recovery buffer (5 minutes):** if a cell failed or ran out of order, use RESET
and work from the top, keeping your predictions. Otherwise, check your answers
and saved outputs. The planned sections take 45 minutes, leaving this buffer.

**Submit:** complete every answer, run the version fill-in, and keep cell outputs saved. Click Share → General
access → Anyone with the link → Viewer → Copy link. Paste the URL into the
Canvas activity by its stated deadline. These files disappear with the runtime;
your saved notebook, outputs, and written evidence are what Dana can inspect later.